In [3]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"hourly": "temperature_2m",
	"models": "meteofrance_seamless",
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 52.52000045776367°N 13.40999984741211°E
Elevation: 38.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                          date  temperature_2m
0   2026-03-03 00:00:00+00:00          4.7195
1   2026-03-03 01:00:00+00:00          4.0695
2   2026-03-03 02:00:00+00:00          3.3695
3   2026-03-03 03:00:00+00:00          3.0195
4   2026-03-03 04:00:00+00:00          2.3695
..                        ...             ...
163 2026-03-09 19:00:00+00:00             NaN
164 2026-03-09 20:00:00+00:00             NaN
165 2026-03-09 21:00:00+00:00             NaN
166 2026-03-09 22:00:00+00:00             NaN
167 2026-03-09 23:00:00+00:00             NaN

[168 rows x 2 columns]


In [ ]:
import requests
import os

path = os.getenv("DOCKER_INFLUXDB_INIT_ADMIN_TOKEN_FILE")
with open (path, "r") as f:
    token = f.read().strip()
print(token)

url = "http://localhost:8086/api/v2/write"
headers = {"Authorization": f"Bearer {token}", 
           "Content-Type": "text/plain, charset=utf-8", 
           "Accept": "application/json"}
params = {"bucket":"get-started"}



In [6]:
hourly_dataframe["timestamp"] = hourly_dataframe["date"].astype("int64") // 10**9

def row_to_line(row): 
    measurement = "weather" 
    temp = row["temperature_2m"] 
    ts = int(row["timestamp"]) 
    return f"{measurement} temperature_2m={temp} {ts}"

lines = "\n".join(hourly_dataframe.apply(row_to_line, axis=1)) 
print(lines)

weather temperature_2m=4.719499588012695 1772496000
weather temperature_2m=4.069499969482422 1772499600
weather temperature_2m=3.369499921798706 1772503200
weather temperature_2m=3.0195000171661377 1772506800
weather temperature_2m=2.369499921798706 1772510400
weather temperature_2m=1.7195000648498535 1772514000
weather temperature_2m=2.2695000171661377 1772517600
weather temperature_2m=4.019499778747559 1772521200
weather temperature_2m=6.019499778747559 1772524800
weather temperature_2m=7.969499588012695 1772528400
weather temperature_2m=10.519499778747559 1772532000
weather temperature_2m=12.469499588012695 1772535600
weather temperature_2m=13.969499588012695 1772539200
weather temperature_2m=15.069499969482422 1772542800
weather temperature_2m=15.269499778747559 1772546400
weather temperature_2m=15.369500160217285 1772550000
weather temperature_2m=14.619500160217285 1772553600
weather temperature_2m=14.269499778747559 1772557200
weather temperature_2m=12.169499397277832 1772560800


In [7]:
response = requests.post(url, headers=headers, params=params, data=lines)

ConnectionError: HTTPConnectionPool(host='localhost', port=8086): Max retries exceeded with url: /api/v2/write?bucket=get-started (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x749baabb4530>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [2]:
import os
from influxdb_client_3 import (
  InfluxDBClient3, InfluxDBError, Point, WritePrecision,
  WriteOptions, write_client_options)

host = "http://localhost:8181"
token = os.getenv('INFLUXDB3_AUTH_TOKEN')
database = "mybucket"

# Create an array of points with tags and fields.
points = [Point("home")
            .tag("room", "Kitchen")
            .field("temp", 25.3)
            .field('hum', 20.2)
            .field('co', 9)]

# With batching mode, define callbacks to execute after a successful or
# failed write request.
# Callback methods receive the configuration and data sent in the request.
def success(self, data: str):
    print(f"Successfully wrote batch: data: {data}")

def error(self, data: str, exception: InfluxDBError):
    print(f"Failed writing batch: config: {self}, data: {data} due: {exception}")

def retry(self, data: str, exception: InfluxDBError):
    print(f"Failed retry writing batch: config: {self}, data: {data} retry: {exception}")

# Configure options for batch writing.
write_options = WriteOptions(batch_size=500,
                                    flush_interval=10_000,
                                    jitter_interval=2_000,
                                    retry_interval=5_000,
                                    max_retries=5,
                                    max_retry_delay=30_000,
                                    exponential_base=2)

# Create an options dict that sets callbacks and WriteOptions.
wco = write_client_options(success_callback=success,
                          error_callback=error,
                          retry_callback=retry,
                          write_options=write_options)

# Instantiate a synchronous instance of the client with your
# InfluxDB credentials and write options, such as Gzip threshold, default tags,
# and timestamp precision. Default precision is nanosecond ('ns').
with InfluxDBClient3(host=host,
                        token=token,
                        database=database,
                        write_client_options=wco) as client:

      client.write(points, write_precision='s')

The retriable error occurred during request. Reason: '<urllib3.connection.HTTPConnection object at 0x76cb1951de50>: Failed to establish a new connection: [Errno 111] Connection refused'.


Failed retry writing batch: config: ('mybucket', 'default', 'ns'), data: b'home,room=Kitchen co=9i,hum=20.2,temp=25.3' retry: <urllib3.connection.HTTPConnection object at 0x76cb1951de50>: Failed to establish a new connection: [Errno 111] Connection refused


The retriable error occurred during request. Reason: '<urllib3.connection.HTTPConnection object at 0x76cb1951fa10>: Failed to establish a new connection: [Errno 111] Connection refused'.


Failed retry writing batch: config: ('mybucket', 'default', 'ns'), data: b'home,room=Kitchen co=9i,hum=20.2,temp=25.3' retry: <urllib3.connection.HTTPConnection object at 0x76cb1951fa10>: Failed to establish a new connection: [Errno 111] Connection refused


The retriable error occurred during request. Reason: '<urllib3.connection.HTTPConnection object at 0x76cb19549640>: Failed to establish a new connection: [Errno 111] Connection refused'.


...

Failed retry writing batch: config: ('mybucket', 'default', 'ns'), data: b'home,room=Kitchen co=9i,hum=20.2,temp=25.3' retry: <urllib3.connection.HTTPConnection object at 0x76cb184ad760>: Failed to establish a new connection: [Errno 111] Connection refused


The retriable error occurred during request. Reason: '<urllib3.connection.HTTPConnection object at 0x76cb184ac650>: Failed to establish a new connection: [Errno 111] Connection refused'.


Failed retry writing batch: config: ('mybucket', 'default', 'ns'), data: b'home,room=Kitchen co=9i,hum=20.2,temp=25.3' retry: <urllib3.connection.HTTPConnection object at 0x76cb184ac650>: Failed to establish a new connection: [Errno 111] Connection refused


The batch item wasn't processed successfully because: HTTPConnectionPool(host='localhost', port=8181): Max retries exceeded with url: /api/v2/write?org=default&bucket=mybucket&precision=ns (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x76cb184aed80>: Failed to establish a new connection: [Errno 111] Connection refused'))


Failed writing batch: config: ('mybucket', 'default', 'ns'), data: b'home,room=Kitchen co=9i,hum=20.2,temp=25.3' due: HTTPConnectionPool(host='localhost', port=8181): Max retries exceeded with url: /api/v2/write?org=default&bucket=mybucket&precision=ns (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x76cb184aed80>: Failed to establish a new connection: [Errno 111] Connection refused'))
